# 04 Physics and model optimization

## Learning outcome

Explain the finite search budget, train-only guarded validation, and why configurations are compared on the same support.

## Method and notation

Three held-out blocks inside the training record test each configuration. Both boundaries are guarded by 1,792 samples, and independent segments reset their dynamics. The final development record never enters candidate selection. Inspect search tables to distinguish alternatives, controls, and the actual selected configuration.

Prerequisites are listed in the project README. This notebook reads the configured run directory. The tables and measurements below come from the recorded run or are recomputed in Python. Use `scripts/execute_notebooks.py --mode full --run-id NEW --data-dir PATH` to regenerate the shared pipeline.

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display
from gearbox_dhon.data import find_input, load_record

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
run_id = os.getenv('GEARBOX_RUN_ID', 'development')
run = root / 'outputs' / 'runs' / run_id
candidate = Path(os.getenv('GEARBOX_DATA_DIR', str(root / 'data' / 'raw')))
data_dir = candidate if candidate.is_dir() and find_input(candidate, 'sigtrain') else None

def read(relative):
    path = run / relative
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def show_files(directory, patterns, limit):
    candidates = [p for p in sorted(directory.rglob('*'))
                  if p.suffix in {'.csv', '.json'} and any(k in p.stem.lower() for k in patterns)]
    for path in candidates[:limit]:
        print(path.relative_to(root))
        if path.suffix == '.csv':
            display(pd.read_csv(path).head(8))
        else:
            value = json.loads(path.read_text())
            print(json.dumps(value, indent=2)[:3600])

def show_figures(patterns, limit):
    candidates = [p for p in sorted((run / 'figures').rglob('*.png'))
                  if any(k in p.stem.lower() for k in patterns)]
    for path in candidates[:limit]:
        print(path.relative_to(root))
        display(Image(filename=str(path), width=690))


# Train-only search for discrete geometry and continuous dynamics

Three guarded training intervals yield 7200 scored targets in total when all three folds complete. They span different local regimes but share one acquisition. For target block $[a,b)$, fit on raw samples in $[0,a-1792)$ and $[b+1792,N)$ as two separate blocks. Warm each fit block for 768 samples and drop that section from the optimizer. The validation prediction for the held-out interval may use measured vibration through the previous target as required by one-step evaluation. These observations never fit features, optimizer parameters, initialization or the adaptation settings.

The finite design covers lag-only, RPM conditioning and angular history controls, then a fixed bank readout and matched trainable variants. Explore compact short-memory banks, a 12-harmonic bank, and a focused mixed order/fixed-Hz bank. Compare exact versus learned carriers, decay-only and frequency-only, fixed-Hz only, innovation and acceleration removal, ridge and head-gate strength, long raw lags, paired ridge augmentation and two explicit adapter variants. The best combined oscillator model uses a train-only objective. The archive-informed geometry is prior engineering evidence. A preliminary sanity fit was also scored on development validation during implementation, so all official validation should be interpreted as development evidence. The runner chose its final configurations from guarded fold ranks without consulting the validation signal.

The selection objective is $J=\mathrm{MSE}_{clean}+0.12\mathrm{MSE}_{stress}+0.004D_{pulse}$ averaged equally across folds. Stress MSE averages +/−1.2% RPM bias with independently seeded 0.2% filtered jitter. The pulse test adds a synthetic once-per-revolution 930 Hz burst. $D_{pulse}$ is the ratio of squared changes in prediction to energy of the injected waveform. Lower distortion reduces modeled leakage into the transient. A separate projection of residual pulse response onto the injected pulse gives amplitude preservation along its original shape. These fixture metrics do not measure sensitivity to a verified cracked gear.

Selection first ranks configurations with both physical parameter groups trainable and a nonlinear head by $J$. The alternative prefers a second-ranked model no larger than the primary with the same gain when available. The rank table retains fixed models even when they have lower utility. The finalists are refit with frozen geometry and scored at indices $[1792,24000)$ on official validation. Three seeds for the primary measure limited initialization variation. Chronological reruns use only earlier fit blocks; they quantify the consequences of extrapolating within this one record. Width, frequency, damping and source geometry interact, so inspect controlled primary-matched variants before attributing change.

The search is an enumerated comparison of a modest finite design. It does not establish global, exhaustive or convergence-based optimality. The high-rate impulsive simulation models a damped event once per turn and is deliberately simple. Fold-average accuracy, robustness, model cost and pulse response must be read together. Real development records have separate label and overlap rules. Final independent datasets remain a separate evaluation.

The run manifest records candidate hyperparameters, the fold and warm-up policy, seed, device and a SHA-256 signature of code, training arrays and experiment design. Per-fold result, predictions and learning trajectories are written separately for safe resume. The official prediction file stores full aligned base and adapted predictions for checking correction separately. The package script consumes these same artifacts to regenerate the plots and checkpoints.


In [ ]:
# Show the full finite search source path and preview the actual scores.
show_files(run / 'search', ('trial', 'fold', 'history', 'config', 'selection'), limit=10)
show_figures(('search', 'pareto', 'trial', 'learn'), limit=5)
show_files(run / 'checkpoints', ('config', 'score'), limit=6)


## Interpretation

Review data scope and limitations in the [course report](../docs/final_course_report.md) and [companion guide](../docs/companion_guide.md). The executed values are attributable to the named run and the exact input and checkpoint hashes in its manifest.